In [0]:
%run /Shared/insclm_capstone/NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime

spark.sql("CREATE DATABASE IF NOT EXISTS silver_insclm")
print("✅ silver_insclm database ready")

✅ silver_insclm database ready


In [0]:
print("\n📥 Loading bronze_policy_master...")
bronze_policy = spark.table(
    "bronze_insclm.bronze_policy_master")
print(f"   Rows: {bronze_policy.count():,}  (expected 1,500)")
bronze_policy.printSchema()


📥 Loading bronze_policy_master...
   Rows: 1,500  (expected 1,500)
root
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- policy_type: string (nullable = true)
 |-- coverage_amount: decimal(18,2) (nullable = true)
 |-- premium_amount: decimal(18,2) (nullable = true)
 |-- policy_status: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _pipeline_run_id: string (nullable = true)



In [0]:
# Columns that trigger a new SCD version when changed
SCD_COLS = ["policy_type", "coverage_amount",
            "premium_amount", "policy_status"]

# MD5 hash of tracked columns — detects any change
hash_expr = F.md5(
    F.concat_ws("||",
        *[F.col(c).cast("string") for c in SCD_COLS]))

source_df = (bronze_policy
    .withColumn("record_hash",    hash_expr)
    .withColumn("effective_date", F.col("last_updated"))
    .withColumn("expiry_date",
        F.lit("9999-12-31 00:00:00")
         .cast(TimestampType()))
    .withColumn("is_current",     F.lit(True))
    .withColumn("created_at",     F.current_timestamp())
    .withColumn("updated_at",     F.current_timestamp()))

print("✅ Source dataframe prepared")
print(f"   Rows: {source_df.count():,}")

✅ Source dataframe prepared
   Rows: 1,500


In [0]:
TABLE_NAME = "silver_insclm.silver_policy_dim"

if not spark.catalog.tableExists(TABLE_NAME):
    print(f"\n📥 Table does not exist — initial load...")

    initial_df = (source_df
        .withColumn("policy_sk",
            F.monotonically_increasing_id())
        .select(
            "policy_sk",
            "policy_id",
            "customer_id",
            "policy_type",
            "coverage_amount",
            "premium_amount",
            "policy_status",
            "start_date",
            "end_date",
            "effective_date",
            "expiry_date",
            "is_current",
            "record_hash",
            "created_at",
            "updated_at"
        ))

    initial_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TABLE_NAME)

    count = spark.table(TABLE_NAME).count()
    print(f"✅ Initial load complete → {count:,} rows")

else:
    print(f"\n📥 Table exists — running SCD MERGE...")
    target      = DeltaTable.forName(spark, TABLE_NAME)
    target_df   = target.toDF()
    current_rows = target_df.filter(
        F.col("is_current") == True)

    # Find policies where tracked columns changed
    changed = (source_df.alias("s")
        .join(current_rows.alias("t"),
              "policy_id", "inner")
        .filter(F.col("s.record_hash") !=
                F.col("t.record_hash"))
        .select(F.col("s.policy_id")))

    changed_count = changed.count()
    print(f"   Changed policies: {changed_count:,}")

    if changed_count > 0:
        # Step 1: Expire old rows
        (target.alias("t")
            .merge(changed.alias("c"),
                "t.policy_id = c.policy_id "
                "AND t.is_current = true")
            .whenMatchedUpdate(set={
                "is_current":  "false",
                "expiry_date": "current_timestamp()",
                "updated_at":  "current_timestamp()"
            })
            .execute())
        print("✅ Old rows expired")

        # Step 2: Insert new versions
        max_sk = target_df.agg(
            F.max("policy_sk")).first()[0] or 0

        new_versions = (source_df
            .join(changed, "policy_id", "inner")
            .withColumn("policy_sk",
                F.monotonically_increasing_id() +
                F.lit(max_sk + 1))
            .select(
                "policy_sk", "policy_id", "customer_id",
                "policy_type", "coverage_amount",
                "premium_amount", "policy_status",
                "start_date", "end_date",
                "effective_date", "expiry_date",
                "is_current", "record_hash",
                "created_at", "updated_at"))

        new_versions.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(TABLE_NAME)

        print(f"✅ {new_versions.count():,} new rows inserted")
    else:
        print("   No changes — policy_dim up to date")


📥 Table does not exist — initial load...
✅ Initial load complete → 1,500 rows


In [0]:
# Verify SCD results
policy_dim = spark.table(TABLE_NAME)
total      = policy_dim.count()
current    = policy_dim.filter(
    F.col("is_current") == True).count()
historical = policy_dim.filter(
    F.col("is_current") == False).count()

print("\n" + "=" * 55)
print("SCD TYPE 2 VERIFICATION")
print("=" * 55)
print(f"Total rows       : {total:,}")
print(f"Current rows     : {current:,}  (expected 1,500)")
print(f"Historical rows  : {historical:,}")

# Check no duplicates
dupes = (policy_dim
    .filter(F.col("is_current") == True)
    .groupBy("policy_id")
    .count()
    .filter(F.col("count") > 1)
    .count())

print(f"Duplicate check  : {dupes}  (expected 0)")

if dupes == 0:
    print("✅ SCD integrity passed")
else:
    print("❌ Duplicate current rows — investigate!")

# Show sample
print("\nSample rows:")
policy_dim.select(
    "policy_id", "policy_type",
    "coverage_amount", "policy_status",
    "effective_date", "expiry_date",
    "is_current", "policy_sk"
).orderBy("policy_id").show(5, truncate=False)

print("=" * 55)
print("✅ NB_04 complete.")
print("   Go back to NB_02 and run Cells 5, 6, 7")


SCD TYPE 2 VERIFICATION
Total rows       : 1,500
Current rows     : 1,500  (expected 1,500)
Historical rows  : 0
Duplicate check  : 0  (expected 0)
✅ SCD integrity passed

Sample rows:
+---------+-----------+---------------+-------------+-------------------+-------------------+----------+---------+
|policy_id|policy_type|coverage_amount|policy_status|effective_date     |expiry_date        |is_current|policy_sk|
+---------+-----------+---------------+-------------+-------------------+-------------------+----------+---------+
|POL000001|Travel     |8498893.00     |Cancelled    |2025-01-06 23:11:21|9999-12-31 00:00:00|true      |0        |
|POL000002|Home       |5067770.50     |Active       |2026-02-24 03:27:22|9999-12-31 00:00:00|true      |1        |
|POL000003|Travel     |3952137.25     |Lapsed       |2025-09-20 10:01:15|9999-12-31 00:00:00|true      |2        |
|POL000004|Home       |2411665.75     |Active       |2025-09-22 20:56:59|9999-12-31 00:00:00|true      |3        |
|POL00000